# 🔬 Training YOLOv8n — ElectroCom61 (Raspberry Pi 4 + Pi Camera Rev 1.3, Model NCNN)

Notebook ini melatih model **YOLOv8n (Nano)** untuk mendeteksi komponen elektronik pada dataset **ElectroCom61**, lalu mengekspornya ke format **NCNN** supaya bisa dijalankan di **Raspberry Pi 4** dengan kamera **Raspberry Pi Camera Rev 1.3**. YOLOv8n dipilih karena paling ringan di keluarga YOLOv8, dan NCNN dipilih karena framework ini dibuat untuk CPU ARM seperti yang ada di Raspberry Pi.

Sebelum training, dataset diperiksa dan dirapikan dulu. Alurnya:

1. **Periksa dataset asli** (Cell 3–7): gambar rusak, label yang tidak valid, gambar tanpa bounding box, gambar kembar, dan kelas yang kosong di data `valid`/`test`. Dataset asli tidak pernah diubah.
2. **Rapikan** (Cell 8–10): label diperbaiki, gambar kembar dibuang, lalu semua gambar berlabel dibagi ulang menjadi train, valid, dan test (70% : 20% : 10%) dengan cara yang menjamin setiap kelas punya contoh di ketiganya. Hasilnya disimpan di folder baru `dataset_electrocom_clean`.
3. **Beri bounding box** pada gambar yang tadinya tidak punya (Cell 11–12), atau keluarkan dari dataset kalau memang tidak berhasil.
4. **Verifikasi dataset bersih** (Cell 13–14) sebelum training dimulai.
5. **Training** (Cell 15–18).
6. **Export ke NCNN** dan semua evaluasi memakai model NCNN itu (Cell 19–24), sehingga angka yang dilaporkan adalah angka dari model yang benar-benar dipakai di Raspberry Pi.

## Cell 1 — Import Library & Pengecekan GPU

Cell pertama menyiapkan semua library yang dipakai di sepanjang notebook dan memeriksa apakah komputer punya GPU yang bisa dipakai PyTorch.

Library yang diimpor dan fungsinya:
- `ultralytics` (`YOLO`): library utama untuk memuat, melatih, mengevaluasi, memprediksi, dan mengekspor model YOLOv8.
- `pathlib` (`Path`): mengelola lokasi file dan folder dengan cara yang rapi, termasuk di Windows.
- `PIL` (`Image`) dan `matplotlib`: membuka gambar dan menampilkannya langsung di notebook, termasuk menggambar bounding box.
- `numpy`: perhitungan angka, dipakai misalnya untuk menghitung akurasi dari confusion matrix.
- `yaml`: membaca dan menulis file konfigurasi dataset (`data.yaml`).
- `torch`: dipakai untuk memeriksa ketersediaan GPU (CUDA).
- `shutil`, `hashlib`, `csv`, `random`, dan `collections`: utilitas bawaan Python untuk menyalin file, mengenali gambar kembar lewat hash, menulis laporan CSV, mengacak data, dan menghitung jumlah.

Notebook juga mengunci **random seed** ke 42. Pembagian data dan pemilihan contoh gambar memakai angka acak, dan dengan seed yang tetap, hasilnya akan sama setiap kali notebook dijalankan ulang.

Setelah itu notebook mencetak versi PyTorch, status CUDA, dan nama GPU. Variabel `DEVICE` diisi `0` (GPU pertama) kalau CUDA tersedia, atau `"cpu"` kalau tidak. Training sebaiknya dilakukan di GPU karena di CPU prosesnya jauh lebih lambat. Raspberry Pi 4 baru dipakai nanti untuk menjalankan model yang sudah jadi, bukan untuk training.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import yaml
import torch
import shutil
import random
import hashlib
import csv
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch       :", torch.__version__)
print("CUDA tersedia :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA PyTorch  :", torch.version.cuda)
    DEVICE = 0
else:
    print("GPU tidak tersedia. Training akan menggunakan CPU (sangat lambat).")
    DEVICE = "cpu"

## Cell 2 — Fungsi Menampilkan Gambar Evaluasi

Setiap kali training, validasi, atau test dijalankan, Ultralytics menyimpan sejumlah grafik evaluasi dalam bentuk file gambar (`.png`) di folder hasilnya. Cell ini membuat fungsi bantu `tampilkan_gambar_evaluasi(folder, judul)` supaya grafik-grafik itu langsung tampil di notebook tanpa perlu dibuka satu per satu dari folder.

Fungsi ini mencari file berikut di folder yang diberikan, lalu menampilkan yang ada:
- `results.png`: grafik hasil per epoch selama training (loss dan metrik). File ini hanya dibuat di folder training.
- `confusion_matrix.png` dan `confusion_matrix_normalized.png`: tabel yang menunjukkan kelas mana yang diprediksi benar dan kelas mana yang tertukar dengan kelas lain. Versi normalized menampilkannya dalam bentuk proporsi, jadi lebih mudah dibandingkan antar kelas.
- `PR_curve.png`: kurva precision terhadap recall untuk tiap kelas. Luas area di bawah kurva berkaitan dengan nilai mAP@50.
- `P_curve.png` dan `R_curve.png`: perubahan precision dan recall ketika batas confidence dinaikkan atau diturunkan.
- `F1_curve.png`: nilai F1 (gabungan precision dan recall) pada berbagai batas confidence. Kurva ini membantu memilih batas confidence yang seimbang.

Kalau sebuah file tidak ada di folder itu, file tersebut dilewati. Kalau tidak ada satu pun yang ditemukan, fungsi mencetak pesan bahwa gambar evaluasi belum tersedia.

In [ ]:
def tampilkan_gambar_evaluasi(folder, judul):
    """Menampilkan file gambar evaluasi yang tersedia di folder hasil YOLO."""
    folder = Path(folder)
    nama_file = [
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "PR_curve.png",
        "P_curve.png",
        "R_curve.png",
        "F1_curve.png",
    ]

    print(f"\n=== {judul} ===")
    ditemukan = 0

    for nama in nama_file:
        file_gambar = folder / nama
        if file_gambar.exists():
            ditemukan += 1
            img = Image.open(file_gambar).convert("RGB")
            plt.figure(figsize=(10, 7))
            plt.imshow(img)
            plt.axis("off")
            plt.title(nama)
            plt.show()

    if ditemukan == 0:
        print("Belum ada gambar evaluasi yang ditemukan pada folder:", folder)

print("Fungsi tampilkan_gambar_evaluasi() siap digunakan.")

## Cell 3 — Menentukan Lokasi Dataset dan Folder Keluaran

Cell ini menentukan semua lokasi folder yang dipakai notebook:

- `RAW_DATASET_DIR`: dataset ElectroCom61 asli yang kamu unduh. Folder ini **hanya dibaca**. Semua perbaikan dilakukan pada salinan, jadi dataset asli tetap utuh dan bisa dipakai lagi kalau notebook perlu diulang dari awal.
- `CLEAN_DATASET_DIR`: folder `dataset_electrocom_clean`, tempat dataset hasil perapian akan dibuat di Cell 10. Isinya nanti tersusun seperti yang diharapkan YOLO: `train`, `valid`, dan `test`, masing-masing berisi `images` dan `labels`, ditambah file `data_clean.yaml`.
- `PROJECT_DIR`: folder `electrocom61_yolov8n_runs`, tempat hasil training, evaluasi, dan prediksi disimpan.
- `MODELS_DIR`: folder `models`, tempat model NCNN dan daftar kelas akan disalin pada Cell 19. Folder inilah yang nanti dibawa ke Raspberry Pi.

Dataset asli harus punya folder `train`, `valid`, `test` (masing-masing dengan `images` dan `labels`) dan file `data.yaml`. Notebook memeriksa folder dan file itu lebih dulu, dan langsung berhenti dengan pesan error yang jelas kalau tidak ditemukan, supaya kesalahan path ketahuan di awal. Kalau dataset kamu ada di lokasi lain, ubah nilai `RAW_DATASET_DIR` di cell ini.

In [ ]:
# Dataset asli (hanya dibaca, tidak pernah diubah)
RAW_DATASET_DIR = Path(
    r"D:\Downloads\ElectroCom61\ElectroCom61 A Multiclass Dataset for Detection of Electronic Components\ElectroCom-61_v2"
)
raw_yaml_path = RAW_DATASET_DIR / "data.yaml"

# Dataset bersih hasil perapian (dibuat di Cell 10)
CLEAN_DATASET_DIR = Path.cwd() / "dataset_electrocom_clean"
clean_yaml_path   = CLEAN_DATASET_DIR / "data_clean.yaml"

# Folder hasil training/evaluasi dan folder model untuk Raspberry Pi
PROJECT_DIR = Path.cwd() / "electrocom61_yolov8n_runs"
MODELS_DIR  = Path.cwd() / "models"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
splits = ["train", "valid", "test"]

print("Dataset asli ditemukan :", RAW_DATASET_DIR.exists())
print("data.yaml ditemukan    :", raw_yaml_path.exists())
print("Dataset asli           :", RAW_DATASET_DIR)
print("Dataset bersih (nanti) :", CLEAN_DATASET_DIR)
print("Folder hasil training  :", PROJECT_DIR)
print("Folder model           :", MODELS_DIR)

if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError("Dataset tidak ditemukan. Periksa kembali nilai RAW_DATASET_DIR.")

if not raw_yaml_path.exists():
    raise FileNotFoundError("data.yaml tidak ditemukan di folder dataset.")

print("\nDataset berhasil ditemukan! ✅")

## Cell 4 — Membaca data.yaml

`data.yaml` adalah file konfigurasi dataset untuk YOLO. Cell ini membukanya dengan library `yaml`, lalu mengambil daftar nama kelas dan jumlahnya.

- `names`: daftar nama kelas komponen elektronik. Urutannya penting, karena angka `class_id` di file label menunjuk ke posisi nama di daftar ini. Misalnya `class_id` 0 berarti nama pertama dalam daftar. Kalau `names` ternyata tersimpan dalam bentuk kamus (`{0: nama, 1: nama, ...}`), cell ini mengubahnya menjadi daftar berurutan.
- `nc`: jumlah kelas yang tertulis di file. Notebook membandingkannya dengan jumlah nama yang sebenarnya. Kalau keduanya berbeda, jumlah nama yang dipakai dan notebook mencetak peringatan.

Hasilnya disimpan di `CLASS_NAMES` dan `NUM_CLASSES`, yang dipakai di hampir semua cell berikutnya: memeriksa `class_id`, menampilkan nama kelas pada gambar, membuat `data_clean.yaml`, dan menyimpan `classes.txt` untuk aplikasi di Raspberry Pi.

In [ ]:
with open(raw_yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

nama_mentah = data["names"]
if isinstance(nama_mentah, dict):
    nama_mentah = [nama_mentah[k] for k in sorted(nama_mentah)]

CLASS_NAMES = list(nama_mentah)
NUM_CLASSES = len(CLASS_NAMES)

print("nc di data.yaml :", data.get("nc"))
print("Jumlah nama     :", NUM_CLASSES)

if data.get("nc") != NUM_CLASSES:
    print("⚠️  nc dan jumlah nama berbeda. Yang dipakai adalah jumlah nama:", NUM_CLASSES)

print("\nDaftar kelas:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:2d}: {name}")

## Cell 5 — Mengecek Jumlah Gambar dan Label (Dataset Asli)

Dataset dibagi menjadi tiga bagian, dan masing-masing punya peran berbeda:
- `train`: data untuk melatih model.
- `valid`: data untuk memantau performa selama training dan memilih bobot terbaik.
- `test`: data untuk pengujian akhir, yang tidak dipakai saat training.

Cell ini menghitung jumlah file gambar (ekstensi `.jpg`, `.jpeg`, `.png`, `.bmp`, atau `.webp`) dan jumlah file label `.txt` di setiap bagian dataset asli, lalu mencetak totalnya. Setiap gambar seharusnya punya satu file label dengan nama yang sama, jadi jumlah gambar dan label biasanya sama. Kalau ada selisih, penyebabnya dicari di cell berikutnya. Hasil cell ini juga memberi gambaran ukuran tiap bagian dataset sebelum dirapikan.

In [ ]:
split_counts = {}

for split in splits:
    images_path = RAW_DATASET_DIR / split / "images"
    labels_path = RAW_DATASET_DIR / split / "labels"

    images = [
        p for p in images_path.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ] if images_path.exists() else []
    labels = list(labels_path.glob("*.txt")) if labels_path.exists() else []

    split_counts[split] = {"images": len(images), "labels": len(labels)}

    print(f"===== {split.upper()} =====")
    print("Images :", len(images))
    print("Labels :", len(labels))
    print()

total_images = sum(v["images"] for v in split_counts.values())
print("TOTAL IMAGES:", total_images)

## Cell 6 — Audit Kualitas Dataset Asli

Label YOLO ditulis satu objek per baris dengan format:

`class_id x_center y_center width height`

Empat angka terakhir adalah koordinat bounding box yang dinormalisasi terhadap ukuran gambar, jadi nilainya harus berada di antara 0 dan 1. Label yang rusak bisa membuat training berhenti dengan error atau diam-diam menurunkan kualitas model, jadi cell ini memeriksa seluruh dataset asli sebelum apa pun dilatih.

Setiap gambar di `train`, `valid`, dan `test` diperiksa untuk hal-hal berikut:
- **Gambar rusak**: file gambar tidak bisa dibuka.
- **Tanpa file label**: gambar tidak punya file `.txt` dengan nama yang sama.
- **Label kosong**: file label ada, tetapi tidak berisi satu pun bounding box yang valid. Bersama gambar yang tidak punya file label, inilah gambar **tanpa bounding box**.
- **Baris bermasalah** di dalam label: jumlah nilainya bukan lima, bukan angka, `class_id` di luar rentang `0` sampai `nc - 1`, kotak yang keluar dari tepi gambar (dipotong ke dalam gambar), kotak yang terlalu kecil sampai tidak berarti (sisi kurang dari 0,5% ukuran gambar, dibuang), dan baris yang tertulis dobel (dibuang salah satunya).
- **Label tanpa gambar**: ada file label yang gambarnya tidak ditemukan.

Fungsi `parse_label()` di cell ini membaca satu file label sekaligus memperbaiki baris yang masih bisa diselamatkan. Fungsi yang sama dipakai lagi nanti untuk memverifikasi dataset bersih.

Cell ini juga mencetak **ukuran dan rasio gambar** dataset. Kamera Raspberry Pi Camera Rev 1.3 (sensor OV5647, 5 MP) menghasilkan gambar berbentuk 4:3, jadi berguna untuk tahu seberapa mirip bentuk gambar dataset dengan gambar yang nanti ditangkap kamera. Hasil audit disimpan di variabel `records` untuk dipakai cell berikutnya.

In [ ]:
MIN_BOX_SIDE = 0.005   # sisi kotak minimal (0,5% dari ukuran gambar)

KETERANGAN_MASALAH = {
    "file_label_gagal_dibaca"      : "File label gagal dibaca",
    "jumlah_nilai_bukan_5"         : "Baris dengan jumlah nilai bukan 5",
    "bukan_angka"                  : "Baris berisi nilai yang bukan angka",
    "class_id_tidak_valid"         : "class_id di luar rentang kelas",
    "kotak_keluar_gambar_dipotong" : "Kotak keluar dari gambar (dipotong)",
    "kotak_terlalu_kecil_dibuang"  : "Kotak terlalu kecil (dibuang)",
    "baris_dobel_dibuang"          : "Baris dobel (dibuang)",
}


def parse_label(label_path):
    # Membaca satu file label. Mengembalikan (daftar baris yang sudah diperbaiki, Counter masalah).
    # Baris yang tidak bisa diselamatkan dibuang dan dicatat sebagai masalah.
    rows, masalah, terlihat = [], Counter(), set()

    try:
        isi = Path(label_path).read_text(encoding="utf-8")
    except Exception:
        masalah["file_label_gagal_dibaca"] += 1
        return rows, masalah

    for baris in isi.splitlines():
        baris = baris.strip()
        if not baris:
            continue

        nilai = baris.split()
        if len(nilai) != 5:
            masalah["jumlah_nilai_bukan_5"] += 1
            continue

        try:
            class_id = int(float(nilai[0]))
            xc, yc, w, h = (float(v) for v in nilai[1:])
        except ValueError:
            masalah["bukan_angka"] += 1
            continue

        if not (0 <= class_id < NUM_CLASSES):
            masalah["class_id_tidak_valid"] += 1
            continue

        # Ubah ke sudut kotak, lalu potong agar tetap di dalam gambar
        x1, y1, x2, y2 = xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2
        cx1, cy1, cx2, cy2 = (min(max(v, 0.0), 1.0) for v in (x1, y1, x2, y2))
        if max(abs(cx1 - x1), abs(cy1 - y1), abs(cx2 - x2), abs(cy2 - y2)) > 1e-4:
            masalah["kotak_keluar_gambar_dipotong"] += 1

        lebar, tinggi = cx2 - cx1, cy2 - cy1
        if lebar < MIN_BOX_SIDE or tinggi < MIN_BOX_SIDE:
            masalah["kotak_terlalu_kecil_dibuang"] += 1
            continue

        baris_baru = (
            class_id,
            round((cx1 + cx2) / 2, 6),
            round((cy1 + cy2) / 2, 6),
            round(lebar, 6),
            round(tinggi, 6),
        )
        if baris_baru in terlihat:
            masalah["baris_dobel_dibuang"] += 1
            continue

        terlihat.add(baris_baru)
        rows.append(baris_baru)

    return rows, masalah


records = []
label_tanpa_gambar = []

for split in splits:
    img_dir = RAW_DATASET_DIR / split / "images"
    lbl_dir = RAW_DATASET_DIR / split / "labels"
    if not img_dir.exists():
        print(f"⚠️  Folder tidak ada, dilewati: {img_dir}")
        continue

    gambar = sorted(
        p for p in img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    )
    stem_gambar = {p.stem for p in gambar}

    if lbl_dir.exists():
        for lf in lbl_dir.glob("*.txt"):
            if lf.stem not in stem_gambar:
                label_tanpa_gambar.append((split, lf.name))

    for img in gambar:
        rec = {"split": split, "img": img, "lbl": None, "rows": [],
               "masalah": Counter(), "size": None, "rusak": False}
        try:
            with Image.open(img) as im:
                im.verify()
            with Image.open(img) as im:
                rec["size"] = im.size
        except Exception:
            rec["rusak"] = True

        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.exists():
            rec["lbl"] = lbl
            rec["rows"], rec["masalah"] = parse_label(lbl)

        records.append(rec)

# ---------- Ringkasan per split ----------
print(f"{'Split':<8}{'Gambar':>8}{'Rusak':>8}{'Tanpa file label':>18}{'Label kosong':>15}")
print("-" * 57)
for split in splits:
    rs = [r for r in records if r["split"] == split]
    rusak      = sum(r["rusak"] for r in rs)
    tanpa_file = sum(1 for r in rs if not r["rusak"] and r["lbl"] is None)
    kosong     = sum(1 for r in rs if not r["rusak"] and r["lbl"] is not None and not r["rows"])
    print(f"{split:<8}{len(rs):>8}{rusak:>8}{tanpa_file:>18}{kosong:>15}")

tanpa_bbox_total = sum(1 for r in records if not r["rusak"] and not r["rows"])
print(f"\nGambar tanpa bounding box (total): {tanpa_bbox_total}")
print(f"Label tanpa gambar               : {len(label_tanpa_gambar)}")

# ---------- Masalah di dalam baris label ----------
total_masalah = Counter()
for r in records:
    total_masalah.update(r["masalah"])

print("\nMasalah pada isi label:")
if total_masalah:
    for kode, jumlah in total_masalah.most_common():
        print(f"  {KETERANGAN_MASALAH.get(kode, kode):<42}: {jumlah}")
else:
    print("  Tidak ada masalah pada isi label. ✅")

# ---------- Ukuran dan rasio gambar ----------
ukuran = Counter(r["size"] for r in records if r["size"])
total_ukuran = sum(ukuran.values())
print("\nUkuran gambar yang paling banyak:")
for (w, h), n in ukuran.most_common(5):
    print(f"  {w} x {h}: {n} gambar")

if total_ukuran:
    rasio_43  = sum(n for (w, h), n in ukuran.items() if h and abs(w / h - 4 / 3) < 0.02)
    persegi   = sum(n for (w, h), n in ukuran.items() if h and abs(w / h - 1) < 0.02)
    print(f"\nGambar berbentuk 4:3     : {rasio_43} ({rasio_43 / total_ukuran:.1%})")
    print(f"Gambar berbentuk persegi : {persegi} ({persegi / total_ukuran:.1%})")
    print("Kamera Pi Camera Rev 1.3 menghasilkan gambar 4:3.")

## Cell 7 — Kelas yang Kosong di Setiap Bagian Dataset Asli

Setelah gambar dan label diperiksa, cell ini menghitung berapa objek setiap kelas di `train`, `valid`, dan `test` pada dataset asli, lalu menampilkan **kelas-kelas yang tidak punya satu pun contoh** di tiap bagian.

Ini penting karena kelas yang kosong di `test` tidak bisa dievaluasi: precision, recall, dan mAP-nya tidak punya dasar hitung, sehingga nilai rata-rata seluruh kelas ikut tidak akurat. Kelas yang kosong di `valid` juga merugikan, karena model memilih bobot terbaiknya berdasarkan data `valid` tanpa pernah menguji kelas itu.

Cell ini juga menampilkan kelas yang **total gambarnya kurang dari 3**. Kelas seperti itu secara matematis tidak mungkin punya contoh di ketiga bagian sekaligus tanpa menaruh gambar yang sama di dua bagian (yang akan membuat hasil evaluasi menjadi bocor dan terlalu bagus). Kalau ada, kelas itu akan dilaporkan lagi setelah pembagian ulang di Cell 10.

Perhitungan ini memakai label yang sudah diperbaiki oleh `parse_label()` di Cell 6, jadi baris yang rusak atau dobel tidak ikut dihitung.

In [ ]:
objek_raw = {s: Counter() for s in splits}
gambar_per_kelas_raw = Counter()

for r in records:
    if r["rusak"]:
        continue
    for baris in r["rows"]:
        objek_raw[r["split"]][baris[0]] += 1
    for c in {baris[0] for baris in r["rows"]}:
        gambar_per_kelas_raw[c] += 1

print("Kelas tanpa satu pun contoh pada dataset asli:")
for split in splits:
    kosong = [CLASS_NAMES[k] for k in range(NUM_CLASSES) if objek_raw[split][k] == 0]
    print(f"\n[{split}] {len(kosong)} kelas kosong")
    if kosong:
        print("  " + ", ".join(kosong))

jarang = [(CLASS_NAMES[k], gambar_per_kelas_raw[k]) for k in range(NUM_CLASSES)
          if gambar_per_kelas_raw[k] < 3]
print(f"\nKelas dengan total gambar kurang dari 3: {len(jarang)}")
for nama, n in jarang:
    print(f"  {nama}: {n} gambar")

## Cell 8 — Merapikan Dataset: Buang Gambar Rusak & Gambar Kembar

Cell ini mengumpulkan semua gambar dari ketiga bagian dataset asli menjadi satu kumpulan (*pool*), lalu merapikannya:

- **Gambar rusak** dibuang, karena tidak bisa dipakai untuk training.
- **Gambar kembar persis** dibuang. Cara mengenalinya: setiap file dihitung sidik jarinya (hash MD5), dan dua file dengan isi yang sama persis punya hash yang sama. Kalau ada gambar yang sama persis di `train` dan `test`, model sebenarnya "mengerjakan soal yang sudah pernah dilihat", sehingga hasil test jadi terlalu bagus dan tidak jujur. Dari tiap kelompok kembar, yang dipertahankan adalah salinan dengan bounding box terbanyak.
- **Nama file dibuat unik**, supaya dua gambar dari bagian dataset yang berbeda tidak saling menimpa saat digabung.

Hasilnya dua daftar:
- `pool_berlabel`: gambar yang punya minimal satu bounding box valid, lengkap dengan baris labelnya yang sudah diperbaiki.
- `pool_kosong`: gambar **tanpa bounding box** (tidak punya file label, atau file labelnya kosong). Gambar-gambar ini ditangani khusus di Cell 11.

Hash dihitung dari isi file, jadi gambar yang hanya mirip (misalnya hasil augmentasi atau foto berurutan) tidak dianggap kembar.

In [ ]:
def hash_file(path):
    md5 = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            md5.update(blok)
    return md5.hexdigest()


jumlah_rusak = 0
jumlah_kembar = 0
kembar_lintas_split = 0
sudah_ada = {}

for rec in records:
    rec["dibuang"] = False
    if rec["rusak"]:
        rec["dibuang"] = True
        jumlah_rusak += 1
        continue

    kunci = hash_file(rec["img"])
    if kunci in sudah_ada:
        lama = sudah_ada[kunci]
        jumlah_kembar += 1
        if lama["split"] != rec["split"]:
            kembar_lintas_split += 1
        # pertahankan salinan yang bounding box-nya lebih banyak
        if len(rec["rows"]) > len(lama["rows"]):
            lama["dibuang"] = True
            sudah_ada[kunci] = rec
        else:
            rec["dibuang"] = True
    else:
        sudah_ada[kunci] = rec

nama_terpakai = set()
pool_berlabel, pool_kosong = [], []

for rec in records:
    if rec["dibuang"]:
        continue

    stem = rec["img"].stem
    if stem in nama_terpakai:
        dasar = f"{stem}_{rec['split']}"
        stem = dasar
        nomor = 1
        while stem in nama_terpakai:
            nomor += 1
            stem = f"{dasar}_{nomor}"
    nama_terpakai.add(stem)

    item = {
        "img"        : rec["img"],
        "stem"       : stem,
        "ext"        : rec["img"].suffix,
        "rows"       : rec["rows"],
        "classes"    : {baris[0] for baris in rec["rows"]},
        "split_asal" : rec["split"],
    }
    (pool_berlabel if rec["rows"] else pool_kosong).append(item)

print("Gambar diperiksa                  :", len(records))
print("Gambar rusak (dibuang)            :", jumlah_rusak)
print("Gambar kembar persis (dibuang)    :", jumlah_kembar)
print("  - di antaranya lintas split     :", kembar_lintas_split)
print("-" * 45)
print("Gambar berlabel (siap dibagi)     :", len(pool_berlabel))
print("Gambar tanpa bounding box         :", len(pool_kosong))

## Cell 9 — Melihat Gambar Tanpa Bounding Box

Sebelum memutuskan apa yang dilakukan dengan gambar tanpa bounding box, cell ini menampilkan beberapa contohnya (sampai 8 gambar, dipilih acak dengan seed tetap). Tujuannya melihat langsung apakah gambar-gambar itu **sebenarnya berisi komponen yang lupa diberi kotak**, atau memang gambar kosong tanpa komponen.

Ini penting karena nasibnya berbeda:
- Gambar yang berisi komponen tetapi tidak diberi kotak akan **merusak training**. Model diajari bahwa komponen itu "bukan apa-apa", dan hasilnya recall menurun.
- Gambar yang memang kosong aman dipakai sebagai latar belakang, tetapi tidak membantu model mengenali komponen.

Karena itu gambar tanpa bounding box tidak langsung dimasukkan begitu saja ke training. Di Cell 11 gambar-gambar ini akan diberi bounding box otomatis dengan bantuan model. Kalau modelnya tidak menemukan komponen dengan keyakinan yang cukup, gambarnya dikeluarkan dari dataset.

In [ ]:
def galeri_gambar(daftar, judul, maks=8, kolom=4):
    if not daftar:
        print("Tidak ada gambar untuk ditampilkan.")
        return

    contoh = random.Random(SEED).sample(daftar, min(maks, len(daftar)))
    baris = (len(contoh) + kolom - 1) // kolom

    fig, axes = plt.subplots(baris, kolom, figsize=(4 * kolom, 4 * baris))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")

    for ax, item in zip(axes, contoh):
        ax.imshow(Image.open(item["img"]).convert("RGB"))
        ax.set_title(item["img"].name[:30], fontsize=8)

    fig.suptitle(judul)
    plt.tight_layout()
    plt.show()


print("Gambar tanpa bounding box:", len(pool_kosong))
galeri_gambar(pool_kosong, "Contoh gambar tanpa bounding box")

## Cell 10 — Pembagian Ulang Data: Train 70% · Valid 20% · Test 10%

Cell ini membagi semua gambar berlabel (`pool_berlabel`) menjadi tiga bagian baru dan menyimpannya di folder `dataset_electrocom_clean`. Dataset asli tidak disentuh.

**Kenapa harus dibagi ulang?** Pembagian bawaan dataset membuat beberapa kelas tidak punya contoh di `valid` atau `test` (terlihat di Cell 7). Pembagian acak biasa juga tidak menjamin hal ini teratasi, apalagi satu gambar bisa berisi beberapa jenis komponen sekaligus.

**Cara pembagiannya:**
1. Tiap gambar dicatat kelas-kelas apa saja yang ada di dalamnya.
2. Kelas yang **paling jarang** diproses lebih dulu. Untuk setiap gambar dari kelas itu, notebook memilih bagian (train, valid, atau test) yang paling "kekurangan" kelas tersebut dibanding targetnya (70%, 20%, 10%). Dengan urutan ini kelas yang sedikit contohnya tidak kalah oleh kelas yang banyak contohnya.
3. Setelah itu dilakukan **pemeriksaan perbaikan**: kalau masih ada kelas yang kosong di `test` atau `valid`, notebook memindahkan satu gambar yang berisi kelas itu dari bagian lain. Gambar yang dipindahkan dipilih dengan hati-hati, yaitu yang tidak membuat kelas lain habis di bagian asalnya.
4. Label ditulis ulang dari versi yang sudah diperbaiki di Cell 6 (kotak yang keluar gambar dipotong, baris dobel dan kotak terlalu kecil dibuang).

Satu gambar hanya masuk ke satu bagian, jadi tidak ada kebocoran data antar bagian. Seed tetap 42, sehingga hasilnya sama setiap kali dijalankan.

Setelah gambar disalin, notebook membuat `data_clean.yaml` dan menampilkan jumlah gambar serta objek di tiap bagian. Kalau ada kelas yang tetap tidak bisa tercakup (misalnya karena total gambarnya hanya 1 atau 2), kelas itu dicetak sebagai peringatan, dan tidak ada gambar yang diduplikasi untuk menutupinya.

In [ ]:
RASIO = (0.70, 0.20, 0.10)   # train, valid, test
NAMA_SPLIT = ["train", "valid", "test"]


def stratified_split(samples, rasio, seed):
    # Membagi gambar ke 3 bagian dengan memprioritaskan kelas yang paling jarang.
    rng = random.Random(seed)
    n = len(samples)

    total_kelas = Counter()
    for s in samples:
        for c in s["classes"]:
            total_kelas[c] += 1

    sisa_target_kelas = {c: [r * total_kelas[c] for r in rasio] for c in total_kelas}
    sisa_target_total = [r * n for r in rasio]

    belum = defaultdict(set)
    for i, s in enumerate(samples):
        for c in s["classes"]:
            belum[c].add(i)

    assign = [None] * n
    while any(a is None for a in assign):
        kandidat = [c for c in belum if belum[c]]
        if not kandidat:
            break
        c = min(kandidat, key=lambda k: (len(belum[k]), rng.random()))
        antrean = sorted(belum[c])
        rng.shuffle(antrean)

        for i in antrean:
            tertinggi = max(sisa_target_kelas[c])
            pilihan = [k for k in range(3) if sisa_target_kelas[c][k] == tertinggi]
            if len(pilihan) > 1:
                tertinggi_total = max(sisa_target_total[k] for k in pilihan)
                pilihan = [k for k in pilihan if sisa_target_total[k] == tertinggi_total]
            k = rng.choice(pilihan)

            assign[i] = k
            for cc in samples[i]["classes"]:
                sisa_target_kelas[cc][k] -= 1
                belum[cc].discard(i)
            sisa_target_total[k] -= 1

    return assign


def perbaiki_kelas_kosong(samples, assign):
    # Pastikan tiap kelas punya contoh di valid dan test dengan memindahkan 1 gambar bila perlu.
    cnt = [Counter() for _ in range(3)]
    for i, s in enumerate(samples):
        for c in s["classes"]:
            cnt[assign[i]][c] += 1

    belum_tercakup = []
    for target in (2, 1):   # test dulu, baru valid
        for c in range(NUM_CLASSES):
            if cnt[target][c] > 0:
                continue
            if sum(cnt[k][c] for k in range(3)) == 0:
                continue   # kelas ini tidak punya gambar sama sekali

            pilihan = None
            for src in [0] + [k for k in (1, 2) if k != target]:
                if src == target:
                    continue
                ketat = [i for i, s in enumerate(samples)
                         if assign[i] == src and c in s["classes"]
                         and all(cnt[src][k] >= 2 for k in s["classes"])]
                longgar = [i for i, s in enumerate(samples)
                           if assign[i] == src and c in s["classes"] and cnt[src][c] >= 2]
                kumpulan = ketat or longgar
                if kumpulan:
                    pilihan = (src, min(kumpulan, key=lambda i: (len(samples[i]["classes"]),
                                                                  len(samples[i]["rows"]))))
                    break

            if pilihan is None:
                belum_tercakup.append((c, NAMA_SPLIT[target]))
                continue

            src, i = pilihan
            assign[i] = target
            for k in samples[i]["classes"]:
                cnt[src][k] -= 1
                cnt[target][k] += 1

    return belum_tercakup


assign = stratified_split(pool_berlabel, RASIO, SEED)
belum_tercakup = perbaiki_kelas_kosong(pool_berlabel, assign)

# ---------- Buat folder dataset bersih ----------
if CLEAN_DATASET_DIR.resolve() == RAW_DATASET_DIR.resolve():
    raise RuntimeError("CLEAN_DATASET_DIR tidak boleh sama dengan dataset asli.")
if CLEAN_DATASET_DIR.exists():
    shutil.rmtree(CLEAN_DATASET_DIR)

for nama in NAMA_SPLIT:
    (CLEAN_DATASET_DIR / nama / "images").mkdir(parents=True, exist_ok=True)
    (CLEAN_DATASET_DIR / nama / "labels").mkdir(parents=True, exist_ok=True)

for item, k in zip(pool_berlabel, assign):
    nama = NAMA_SPLIT[k]
    shutil.copy2(item["img"], CLEAN_DATASET_DIR / nama / "images" / f"{item['stem']}{item['ext']}")
    isi = "\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in item["rows"])
    (CLEAN_DATASET_DIR / nama / "labels" / f"{item['stem']}.txt").write_text(isi + "\n", encoding="utf-8")

# ---------- data_clean.yaml ----------
clean_yaml_data = {
    "path" : str(CLEAN_DATASET_DIR.resolve()),
    "train": "train/images",
    "val"  : "valid/images",
    "test" : "test/images",
    "nc"   : NUM_CLASSES,
    "names": CLASS_NAMES,
}
with open(clean_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(clean_yaml_data, f, sort_keys=False, allow_unicode=True)

# ---------- Ringkasan ----------
print(f"{'Bagian':<8}{'Gambar':>8}{'Objek':>8}{'Porsi gambar':>15}")
print("-" * 39)
for k, nama in enumerate(NAMA_SPLIT):
    idx = [i for i, a in enumerate(assign) if a == k]
    objek = sum(len(pool_berlabel[i]["rows"]) for i in idx)
    print(f"{nama:<8}{len(idx):>8}{objek:>8}{len(idx) / len(pool_berlabel):>15.1%}")

print("\nKonfigurasi dataset tersimpan di:", clean_yaml_path)

if belum_tercakup:
    print("\n⚠️  Kelas yang tetap tidak bisa tercakup (gambarnya terlalu sedikit):")
    for c, bagian in belum_tercakup:
        print(f"  - {CLASS_NAMES[c]} tidak punya contoh di {bagian}")
else:
    print("\nSemua kelas punya contoh di train, valid, dan test. ✅")

## Cell 11 — Memberi Bounding Box Otomatis pada Gambar yang Belum Punya

Cell ini menangani gambar yang tidak punya bounding box (`pool_kosong`). Kotak tidak bisa dibuat dengan aturan sederhana, jadi dipakai teknik **pseudo-labeling**: sebuah model "pelabel" dilatih dulu pada gambar yang sudah berlabel benar, lalu model itu diminta menandai komponen pada gambar yang belum berlabel.

Langkah-langkahnya:
1. **Model pelabel dilatih** dari `yolov8n.pt` pada dataset bersih dari Cell 10, memakai pengaturan ringan (`LABELER_CONFIG`: maksimal 30 epoch, early stopping `patience` 8). Model ini hanya alat bantu pelabelan, bukan model akhir.
2. **Setiap gambar tanpa bounding box diprediksi.** Hanya deteksi dengan confidence minimal `PSEUDO_CONF` (0,50) yang diterima. Ambang yang cukup tinggi dipilih supaya kotak yang masuk ke dataset benar-benar yakin, dan kotak yang ragu-ragu tidak mencemari label.
3. **Gambar yang berhasil mendapat minimal satu kotak** disalin ke bagian **train saja**, lengkap dengan labelnya. Gambar ini sengaja tidak dimasukkan ke `valid` maupun `test`, karena labelnya buatan model. Kalau ikut masuk ke data evaluasi, nilai evaluasi akan mengukur seberapa cocok model dengan dirinya sendiri, bukan dengan label manusia.
4. **Gambar yang tetap tidak mendapat kotak dikeluarkan** dari dataset. Gambar yang jelas berisi komponen tetapi tidak berlabel akan mengajari model bahwa komponen itu hanyalah latar belakang.

Kalau tidak ada gambar tanpa bounding box (`pool_kosong` kosong), cell ini langsung dilewati tanpa training apa pun.

Catatan: pada langkah ini bobot `.pt` dari model pelabel dipakai untuk membuat prediksi. Itu hanya bagian dari persiapan data. Seluruh evaluasi dan prediksi untuk hasil akhir memakai model NCNN, mulai Cell 19.

In [ ]:
LABELER_CONFIG = {
    "epochs"    : 30,
    "patience"  : 8,
    "imgsz"     : 640,
    "batch"     : 8,
    "optimizer" : "Adam",
    "lr0"       : 0.001,
    "device"    : DEVICE,
    "workers"   : 2,
    "pretrained": True,
    "plots"     : False,
    "verbose"   : True,
    "project"   : str(PROJECT_DIR),
    "name"      : "labeler_pseudo",
}
PSEUDO_CONF = 0.50

pseudo_items   = []   # gambar yang berhasil diberi bounding box otomatis
laporan_kosong = []   # (nama file, status, jumlah bbox) untuk setiap gambar tanpa bbox

if len(pool_kosong) == 0:
    print("Tidak ada gambar tanpa bounding box. Langkah ini dilewati. ✅")
else:
    print("=" * 60)
    print(f"MELATIH MODEL PELABEL | {len(pool_kosong)} gambar akan diberi bounding box")
    print("=" * 60)

    model_pelabel = YOLO("yolov8n.pt")
    hasil_pelabel = model_pelabel.train(data=str(clean_yaml_path), **LABELER_CONFIG)
    pelabel_terbaik = YOLO(str(Path(hasil_pelabel.save_dir) / "weights" / "best.pt"))

    for item in pool_kosong:
        r = pelabel_terbaik.predict(
            source  = str(item["img"]),
            imgsz   = LABELER_CONFIG["imgsz"],
            conf    = PSEUDO_CONF,
            iou     = 0.5,
            device  = DEVICE,
            verbose = False
        )[0]

        baris_baru, terlihat = [], set()
        if len(r.boxes) > 0:
            xywhn = r.boxes.xywhn.cpu().numpy()
            kelas = r.boxes.cls.cpu().numpy().astype(int)
            for (xc, yc, w, h), c in zip(xywhn, kelas):
                if w < MIN_BOX_SIDE or h < MIN_BOX_SIDE:
                    continue
                baris = (int(c), round(float(xc), 6), round(float(yc), 6),
                         round(float(w), 6), round(float(h), 6))
                if baris not in terlihat:
                    terlihat.add(baris)
                    baris_baru.append(baris)

        if not baris_baru:
            laporan_kosong.append((item["img"].name, "dikeluarkan_dari_dataset", 0))
            continue

        shutil.copy2(item["img"], CLEAN_DATASET_DIR / "train" / "images" / f"{item['stem']}{item['ext']}")
        isi = "\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in baris_baru)
        (CLEAN_DATASET_DIR / "train" / "labels" / f"{item['stem']}.txt").write_text(isi + "\n", encoding="utf-8")

        pseudo_items.append({"stem": item["stem"], "ext": item["ext"], "n": len(baris_baru)})
        laporan_kosong.append((item["img"].name, "diberi_bbox_otomatis", len(baris_baru)))

    print("\nGambar tanpa bbox yang diproses      :", len(pool_kosong))
    print("Berhasil diberi bbox (masuk ke train) :", len(pseudo_items))
    print("Dikeluarkan dari dataset               :", len(pool_kosong) - len(pseudo_items))

## Cell 12 — Memeriksa Hasil Bounding Box Otomatis & Laporan CSV

Label buatan model tetap perlu dilihat manusia. Cell ini melakukan dua hal:

1. **Menampilkan hasil pelabelan otomatis.** Sampai 4 gambar yang tadinya tanpa bounding box ditampilkan lengkap dengan kotak dan nama kelas yang diberikan model, dalam warna oranye supaya mudah dibedakan dari label asli. Perhatikan apakah kotaknya menempel pada komponen yang benar dan apakah nama kelasnya sesuai. Kalau banyak yang meleset, naikkan `PSEUDO_CONF` di Cell 11 (misalnya ke 0,7), lalu jalankan ulang dari Cell 10.
2. **Menyimpan laporan `laporan_gambar_tanpa_bbox.csv`** di folder dataset bersih. Isinya setiap gambar yang tadinya tidak punya bounding box, statusnya (`diberi_bbox_otomatis` atau `dikeluarkan_dari_dataset`), dan jumlah kotak yang diberikan. File ini berguna sebagai catatan penanganan data untuk laporan skripsi.

Cell ini juga mendefinisikan fungsi `gambar_bbox()`, yang menggambar satu gambar lengkap dengan bounding box dari file labelnya. Fungsi yang sama dipakai lagi di Cell 14.

In [ ]:
def gambar_bbox(image_path, label_path, judul, warna="lime"):
    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    jumlah = 0
    if Path(label_path).exists():
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                values = line.strip().split()
                if len(values) != 5:
                    continue
                class_id = int(float(values[0]))
                x_center, y_center, width, height = map(float, values[1:])
                box_w = width  * img_w
                box_h = height * img_h
                x = (x_center * img_w) - box_w / 2
                y = (y_center * img_h) - box_h / 2
                ax.add_patch(patches.Rectangle((x, y), box_w, box_h,
                                               fill=False, linewidth=2, edgecolor=warna))
                ax.text(x, max(0, y - 5), CLASS_NAMES[class_id], fontsize=9, color="white",
                        bbox=dict(facecolor="black", alpha=0.7))
                jumlah += 1
    else:
        print("File label tidak ditemukan:", label_path)

    ax.set_title(f"{judul}: {Path(image_path).name} ({jumlah} objek)")
    ax.axis("off")
    plt.show()


if pseudo_items:
    contoh = random.Random(SEED).sample(pseudo_items, min(4, len(pseudo_items)))
    print(f"Menampilkan {len(contoh)} contoh gambar yang diberi bounding box otomatis.")
    for p in contoh:
        gambar_bbox(
            CLEAN_DATASET_DIR / "train" / "images" / f"{p['stem']}{p['ext']}",
            CLEAN_DATASET_DIR / "train" / "labels" / f"{p['stem']}.txt",
            "Bbox otomatis",
            warna="orange"
        )
else:
    print("Tidak ada gambar yang diberi bounding box otomatis.")

laporan_path = CLEAN_DATASET_DIR / "laporan_gambar_tanpa_bbox.csv"
with open(laporan_path, "w", newline="", encoding="utf-8") as f:
    penulis = csv.writer(f)
    penulis.writerow(["nama_file", "status", "jumlah_bbox"])
    penulis.writerows(laporan_kosong)

print("\nLaporan tersimpan di:", laporan_path)

## Cell 13 — Verifikasi Dataset Bersih & Jumlah Objek per Kelas

Ini pemeriksaan terakhir sebelum training. Seluruh isi `dataset_electrocom_clean` dibaca ulang dengan fungsi `parse_label()` yang sama, lalu notebook memastikan:
- setiap gambar punya file label,
- tidak ada label yang kosong, artinya **setiap gambar punya minimal satu bounding box**,
- tidak ada baris label yang bermasalah,
- setiap kelas punya contoh di `train`, `valid`, dan `test`.

Setelah itu cell ini menampilkan **tabel jumlah objek per kelas**: setiap kelas lengkap dengan ID, nama, jumlah objek di `train`, `valid`, `test`, dan totalnya, dengan baris paling bawah berisi jumlah keseluruhan. Yang dihitung adalah **jumlah objek**, bukan jumlah gambar. Satu gambar bisa berisi beberapa komponen, dan setiap baris label dihitung sebagai satu objek.

Ringkasan di bawah tabel menyebut kelas dengan objek terbanyak dan tersedikit di `train`, serta kelas yang masih kosong di salah satu bagian (kalau ada). Grafik batang menunjukkan jumlah objek per kelas pada data `train`, diurutkan dari yang terbanyak, sehingga terlihat apakah datanya seimbang atau ada kelas yang jauh lebih sedikit dari yang lain. Kelas dengan contoh training sedikit biasanya lebih sulit dikenali model, jadi precision dan recall-nya cenderung lebih rendah.

Hasil hitung gambar dan objek disimpan di `ringkasan_data` untuk ditampilkan lagi di ringkasan akhir (Cell 24).

In [ ]:
jumlah_objek  = {s: Counter() for s in splits}
ringkasan_data = {"gambar": {}, "objek": {}}
masalah_akhir = Counter()
gambar_tanpa_label = 0
gambar_tanpa_bbox  = 0

print(f"{'Bagian':<8}{'Gambar':>8}{'Label':>8}{'Objek':>8}")
print("-" * 32)

for split in splits:
    img_dir = CLEAN_DATASET_DIR / split / "images"
    lbl_dir = CLEAN_DATASET_DIR / split / "labels"
    gambar = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in image_extensions]
    jumlah_label = len(list(lbl_dir.glob("*.txt")))

    for p in gambar:
        lbl = lbl_dir / f"{p.stem}.txt"
        if not lbl.exists():
            gambar_tanpa_label += 1
            continue
        rows, masalah = parse_label(lbl)
        masalah_akhir.update(masalah)
        if not rows:
            gambar_tanpa_bbox += 1
        for baris in rows:
            jumlah_objek[split][baris[0]] += 1

    total_objek_split = sum(jumlah_objek[split].values())
    ringkasan_data["gambar"][split] = len(gambar)
    ringkasan_data["objek"][split]  = total_objek_split
    print(f"{split:<8}{len(gambar):>8}{jumlah_label:>8}{total_objek_split:>8}")

print("\nGambar tanpa file label    :", gambar_tanpa_label)
print("Gambar tanpa bounding box  :", gambar_tanpa_bbox)
print("Baris label bermasalah     :", sum(masalah_akhir.values()))

if gambar_tanpa_label == 0 and gambar_tanpa_bbox == 0 and not masalah_akhir:
    print("Semua gambar punya label dan bounding box yang valid. ✅")

# ---------- Tabel jumlah objek per kelas ----------
print(f"\n{'ID':>3}  {'Kelas':<30}{'Train':>8}{'Valid':>8}{'Test':>8}{'Total':>8}")
print("-" * 65)
for class_id in range(NUM_CLASSES):
    n_train = jumlah_objek["train"][class_id]
    n_valid = jumlah_objek["valid"][class_id]
    n_test  = jumlah_objek["test"][class_id]
    print(f"{class_id:>3}  {CLASS_NAMES[class_id]:<30}{n_train:>8}{n_valid:>8}{n_test:>8}{n_train + n_valid + n_test:>8}")

total_train = sum(jumlah_objek["train"].values())
total_valid = sum(jumlah_objek["valid"].values())
total_test  = sum(jumlah_objek["test"].values())
print("-" * 65)
print(f"{'':>3}  {'TOTAL OBJEK':<30}{total_train:>8}{total_valid:>8}{total_test:>8}{total_train + total_valid + total_test:>8}")

# ---------- Ringkasan ----------
train_per_kelas = {k: jumlah_objek["train"][k] for k in range(NUM_CLASSES)}
terbanyak  = max(train_per_kelas, key=train_per_kelas.get)
tersedikit = min(train_per_kelas, key=train_per_kelas.get)
print(f"\nKelas terbanyak di train : {CLASS_NAMES[terbanyak]} ({train_per_kelas[terbanyak]} objek)")
print(f"Kelas tersedikit di train: {CLASS_NAMES[tersedikit]} ({train_per_kelas[tersedikit]} objek)")

ada_kosong = False
for split in splits:
    kosong = [CLASS_NAMES[k] for k in range(NUM_CLASSES) if jumlah_objek[split][k] == 0]
    if kosong:
        ada_kosong = True
        print(f"⚠️  Kelas tanpa contoh di {split}: {', '.join(kosong)}")
if not ada_kosong:
    print("Tidak ada kelas yang kosong di train, valid, maupun test. ✅")

# ---------- Grafik jumlah objek per kelas (data train) ----------
urutan = sorted(range(NUM_CLASSES), key=lambda k: train_per_kelas[k], reverse=True)
nama   = [CLASS_NAMES[k] for k in urutan]
nilai  = [train_per_kelas[k] for k in urutan]

fig, ax = plt.subplots(figsize=(10, max(6, 0.3 * NUM_CLASSES)))
batang = ax.barh(nama, nilai, color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Jumlah objek")
ax.set_title("Jumlah Objek per Kelas pada Data Train")
for b, v in zip(batang, nilai):
    ax.text(v, b.get_y() + b.get_height() / 2, f" {v}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## Cell 14 — Menampilkan Ground Truth (Contoh Gambar Berlabel)

Ground truth adalah anotasi asli dari dataset, yaitu kotak dan nama kelas yang ditandai pada setiap objek. Cell ini menampilkan 5 gambar acak dari folder `train` pada **dataset bersih**, lengkap dengan anotasinya, memakai fungsi `gambar_bbox()` dari Cell 12.

Caranya: file label gambar dibaca, lalu koordinat YOLO yang berbentuk proporsi (pusat, lebar, tinggi) diubah menjadi koordinat piksel sesuai ukuran gambar. Setiap objek digambar sebagai kotak hijau dengan nama kelasnya di atas kotak.

Tujuan cell ini adalah pemeriksaan visual terakhir sebelum training. Kotak harus menempel pada komponen yang benar, dan nama kelasnya harus sesuai dengan komponennya. Kalau kotak melenceng atau nama kelasnya salah, berarti ada masalah di dataset atau di `data.yaml` yang perlu diperbaiki sebelum training.

In [ ]:
train_images = sorted([
    p for p in (CLEAN_DATASET_DIR / "train" / "images").iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
])

jumlah_contoh = min(5, len(train_images))
contoh_gt = random.Random(SEED).sample(train_images, jumlah_contoh)
print(f"Menampilkan {jumlah_contoh} contoh gambar berlabel dari folder train (dataset bersih).")

for image_path in contoh_gt:
    gambar_bbox(
        image_path,
        CLEAN_DATASET_DIR / "train" / "labels" / f"{image_path.stem}.txt",
        "Ground Truth"
    )

## Cell 15 — Load Model YOLOv8n (Nano)

Cell ini memuat model **YOLOv8n** dari file bobot `yolov8n.pt`. Huruf `n` berarti *nano*, varian YOLOv8 yang paling kecil dan ringan, jadi sesuai untuk perangkat dengan CPU terbatas seperti Raspberry Pi 4. File bobot akan diunduh otomatis kalau belum ada di komputer.

Bobot ini adalah bobot *pretrained*, yaitu model yang sudah dilatih lebih dulu pada dataset besar (COCO) sehingga sudah mengenali ciri dasar gambar seperti tepi, bentuk, dan tekstur. Training di dataset ElectroCom61 nanti tidak mulai dari nol, melainkan menyesuaikan bobot ini supaya mengenali komponen elektronik. Cara ini disebut transfer learning dan biasanya lebih cepat dan lebih stabil dibanding melatih dari awal.

Notebook mencetak pesan berhasil dimuat dan jumlah parameter model sebagai gambaran ukuran modelnya.

In [ ]:
# YOLOv8n (Nano) — ringan, cocok untuk Raspberry Pi 4
model = YOLO("yolov8n.pt")
print("YOLOv8n pretrained berhasil dimuat.")
print(f"Jumlah parameter model: {sum(p.numel() for p in model.model.parameters()):,}")

## Cell 16 — Konfigurasi Training & Export

Cell ini mengumpulkan semua pengaturan training dan export di satu tempat supaya mudah dibaca dan diubah.

**`BASE_CONFIG` (pengaturan umum):**
- `optimizer`: Adam, algoritma yang memperbarui bobot model.
- `lr0`: learning rate awal 0,001, yaitu besar langkah pembaruan bobot di awal training.
- `device`: GPU atau CPU, sesuai hasil pengecekan di Cell 1.
- `workers`: jumlah proses yang memuat data secara paralel.
- `pretrained`: memakai bobot pretrained sebagai titik awal.
- `plots`: menyimpan grafik hasil training.
- `verbose`: menampilkan log training secara rinci.

**`TRAIN_CONFIG` (pengaturan training):**
- `epochs` = 50: batas maksimum jumlah epoch. Satu epoch berarti seluruh data training dilewati satu kali. Training bisa berhenti sebelum epoch ke-50 kalau early stopping aktif.
- `patience` = 10: pengaturan **early stopping**. Setelah setiap epoch, Ultralytics memeriksa apakah performa di data `valid` membaik. Kalau selama 10 epoch berturut-turut tidak ada perbaikan, training dihentikan otomatis dan bobot terbaik tetap tersimpan sebagai `best.pt`. Early stopping menghemat waktu dan mengurangi risiko overfitting.
- `imgsz` = 640: setiap gambar diubah ke ukuran 640 piksel sebelum diproses. Ukuran harus kelipatan 32.
- `batch` = 8: jumlah gambar yang diproses dalam satu langkah. Kalau muncul error *CUDA out of memory*, turunkan ke 4.
- `name`: nama subfolder untuk hasil training ini.

**`EXPORT_CONFIG` (pengaturan export ke NCNN untuk Raspberry Pi 4):**
- `format` = `"ncnn"`: format model untuk CPU ARM yang dipakai Raspberry Pi.
- `imgsz` = 640: ukuran input model NCNN. Ukuran ini **terkunci di dalam model**, jadi di Raspberry Pi gambar harus diproses dengan ukuran yang sama. Semakin kecil ukurannya, semakin cepat model berjalan di Pi tetapi komponen yang kecil makin sulit dikenali. Kalau nanti di Pi kecepatannya terasa kurang, ubah nilai ini (misalnya ke 480 atau 416), lalu jalankan ulang dari Cell 19. Semua angka evaluasi di Cell 20–22 otomatis memakai ukuran yang sama dengan hasil export.
- `half` = `False`: hitungan tetap 32-bit. CPU Raspberry Pi 4 (Cortex-A72) tidak punya dukungan hitungan 16-bit bawaan, jadi mode 16-bit tidak memberi percepatan berarti.

Kamera Raspberry Pi Camera Rev 1.3 memakai sensor OV5647 (5 MP) dengan **fokus tetap**, dan gambarnya berbentuk 4:3. Karena fokusnya tidak bisa diatur, komponen yang terlalu dekat ke lensa bisa tampak buram. Hal ini perlu diingat saat memasang kamera di Raspberry Pi.

In [ ]:
# Konfigurasi dasar
BASE_CONFIG = {
    "optimizer" : "Adam",
    "lr0"       : 0.001,
    "device"    : DEVICE,
    "workers"   : 2,
    "pretrained": True,
    "plots"     : True,
    "verbose"   : True,
}

# Konfigurasi training
TRAIN_CONFIG = {
    "epochs"  : 50,    # batas maksimum epoch
    "patience": 10,    # early stopping: berhenti jika 10 epoch berturut-turut tidak membaik
    "imgsz"   : 640,
    "batch"   : 8,
    "name"    : "yolov8n_50epoch_640",
}

# Konfigurasi export untuk Raspberry Pi 4
EXPORT_CONFIG = {
    "format": "ncnn",
    "imgsz" : 640,
    "half"  : False,
}

print("Konfigurasi training :", TRAIN_CONFIG)
print("Konfigurasi export   :", EXPORT_CONFIG)

## Cell 17 — Training Model

Cell ini menjalankan training. Model YOLOv8n pretrained dimuat, lalu dilatih pada **dataset bersih** (`data_clean.yaml`) memakai pengaturan di `BASE_CONFIG` dan `TRAIN_CONFIG` (maksimal 50 epoch, early stopping dengan `patience` 10, ukuran gambar 640, batch size 8).

Selama training, setiap epoch terdiri dari dua tahap. Pertama, model belajar dari data `train` dan bobotnya diperbarui. Kedua, model dicoba pada data `valid` untuk melihat apakah performanya membaik. Kemajuan tiap epoch ditampilkan di log, berisi nilai loss dan metrik validasi.

Karena early stopping aktif, training berhenti lebih awal kalau performa validasi tidak membaik selama 10 epoch berturut-turut. Di log akan muncul pesan `EarlyStopping` yang menyebut epoch terbaiknya. Yang disimpan sebagai `best.pt` adalah epoch terbaik, bukan epoch saat training berhenti.

Hasil training disimpan di `PROJECT_DIR/yolov8n_50epoch_640`. Di dalam subfolder `weights` ada dua file bobot:
- `best.pt`: bobot dari epoch dengan performa validasi terbaik. File inilah yang diekspor ke NCNN pada Cell 19.
- `last.pt`: bobot dari epoch terakhir.

Folder itu juga berisi grafik hasil training seperti `results.png`. Setelah training selesai, notebook mencetak lokasi folder hasil dan memastikan `best.pt` ada.

> ⏱️ **Waktu training** tergantung komputer, dan bisa lebih singkat kalau early stopping aktif. Jangan tutup Jupyter atau matikan laptop selama proses berjalan.

Ultralytics selalu menyimpan bobot training dalam bentuk `.pt`. File itu hanya menjadi bahan untuk export, dan setelah Cell 19 tidak dipakai lagi.

In [ ]:
print("=" * 60)
print(
    f"TRAINING — YOLOv8n | max {TRAIN_CONFIG['epochs']} epoch | "
    f"patience={TRAIN_CONFIG['patience']} | imgsz={TRAIN_CONFIG['imgsz']} | batch={TRAIN_CONFIG['batch']}"
)
print("=" * 60)

model_train   = YOLO("yolov8n.pt")
results_train = model_train.train(
    data     = str(clean_yaml_path),
    epochs   = TRAIN_CONFIG["epochs"],
    patience = TRAIN_CONFIG["patience"],
    imgsz    = TRAIN_CONFIG["imgsz"],
    batch    = TRAIN_CONFIG["batch"],
    project  = str(PROJECT_DIR),
    name     = TRAIN_CONFIG["name"],
    **BASE_CONFIG
)

run_dir = Path(results_train.save_dir)
best_pt = run_dir / "weights" / "best.pt"

print("\nTraining selesai.")
print("Folder hasil :", run_dir)
print("best.pt      :", best_pt)
print("best.pt ada  :", best_pt.exists())

## Cell 18 — Grafik Hasil Training

Selama training, Ultralytics mencatat nilai loss dan metrik di setiap epoch, lalu menggambarnya dalam satu file `results.png` di folder hasil training. Cell ini menampilkan grafik itu langsung di notebook. Grafik ini menunjukkan bagaimana model belajar dari epoch pertama sampai terakhir, sehingga paling berguna untuk menilai apakah training berjalan sehat.

`results.png` berisi sepuluh grafik kecil. Tiap grafik punya garis hasil per epoch dan garis versi yang sudah dihaluskan agar kecenderungannya lebih mudah dilihat:
- **Loss training** (`train/box_loss`, `train/cls_loss`, `train/dfl_loss`): ukuran kesalahan model pada data train. `box_loss` untuk posisi dan ukuran kotak, `cls_loss` untuk kesalahan kelas, dan `dfl_loss` untuk ketepatan tepi kotak. Nilainya seharusnya turun dari epoch ke epoch.
- **Loss validasi** (`val/box_loss`, `val/cls_loss`, `val/dfl_loss`): tiga loss yang sama, tetapi dihitung pada data `valid`.
- **Metrik validasi** (`metrics/precision(B)`, `metrics/recall(B)`, `metrics/mAP50(B)`, `metrics/mAP50-95(B)`): huruf B berarti bounding box. Nilainya seharusnya naik, lalu mulai mendatar.

Cara membacanya:
- Loss training dan loss validasi sama-sama turun lalu melandai, dan metrik naik lalu mendatar: training berjalan sehat.
- Loss training terus turun tetapi loss validasi mulai naik: tanda overfitting, yaitu model terlalu menghafal data train.
- Metrik masih naik jelas sampai epoch terakhir: model mungkin belum selesai belajar, jadi jumlah epoch bisa ditambah.

Grafik ini bisa dijadikan lampiran di laporan sebagai bukti proses training.

In [ ]:
grafik_training = run_dir / "results.png" if "run_dir" in globals() else None

if grafik_training is not None and grafik_training.exists():
    img = Image.open(grafik_training).convert("RGB")
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Grafik Hasil Training (loss dan metrik per epoch)")
    plt.show()
else:
    print("results.png tidak ditemukan.")
    print("Jalankan Cell 17 (training) terlebih dahulu.")

## Cell 19 — Export Model ke Format NCNN untuk Raspberry Pi 4

NCNN adalah framework inferensi neural network yang dikembangkan khusus untuk perangkat dengan CPU ARM seperti Raspberry Pi 4, tanpa bergantung pada GPU. Karena itu NCNN dipakai sebagai format model akhir di proyek ini.

Cell ini melakukan lima hal:
1. Memuat `best.pt`, bobot terbaik hasil training.
2. Mengekspornya ke format **NCNN** memakai `EXPORT_CONFIG` dari Cell 16. Hasilnya berupa sebuah **folder** (bukan satu file) yang berisi `model.ncnn.param`, `model.ncnn.bin`, dan `metadata.yaml`. Ketiganya harus dibawa bersama ke Raspberry Pi.
3. Menyalin folder itu ke `models/best_ncnn_model`.
4. Menyimpan daftar nama kelas ke `models/classes.txt`, satu nama per baris sesuai urutan `class_id`, untuk dipakai aplikasi deteksi.
5. Memuat model NCNN tadi sebagai `eval_model`. **Model inilah yang dipakai untuk semua evaluasi dan prediksi** di Cell 20 sampai 23, bukan `best.pt`.

Ada satu hal yang berbeda dibanding sebelumnya: evaluasi dijalankan dengan `device="cpu"`. Di Raspberry Pi tidak ada GPU, jadi model NCNN dinilai di CPU juga supaya kondisinya serupa. Angka akurasinya sama dengan yang akan didapat di Pi, tetapi **kecepatannya tidak**. Laptop jauh lebih cepat dari Raspberry Pi 4, jadi kecepatan (FPS) harus diukur langsung di Pi.

Saat export pertama kali, Ultralytics mungkin memasang paket tambahan untuk NCNN secara otomatis, dan itu butuh koneksi internet.

Notebook juga mencetak isi folder model beserta ukurannya sebagai gambaran besar file yang akan dipindahkan ke Raspberry Pi.

In [ ]:
print("=" * 60)
print("EXPORT MODEL KE FORMAT NCNN (RASPBERRY PI 4)")
print("=" * 60)

pt_model  = YOLO(str(best_pt))
ncnn_path = pt_model.export(**EXPORT_CONFIG)

# Salin folder NCNN ke folder models/
src_ncnn  = Path(str(ncnn_path))
dest_ncnn = MODELS_DIR / "best_ncnn_model"
if dest_ncnn.exists():
    shutil.rmtree(dest_ncnn)
shutil.copytree(src_ncnn, dest_ncnn)
print(f"\n✅ Folder model NCNN disimpan ke: {dest_ncnn.resolve()}")

# Simpan daftar nama kelas
classes_path = MODELS_DIR / "classes.txt"
with open(classes_path, "w", encoding="utf-8") as f:
    for nama in CLASS_NAMES:
        f.write(f"{nama}\n")
print(f"✅ Daftar kelas disimpan ke   : {classes_path.resolve()}")

print("\nIsi folder model NCNN:")
for p in sorted(dest_ncnn.iterdir()):
    print(f"  {p.name:<24}{p.stat().st_size / 1e6:>8.2f} MB")

# Model NCNN dipakai untuk semua evaluasi dan prediksi berikutnya
eval_model = YOLO(str(dest_ncnn), task="detect")
print("\nModel NCNN siap dipakai untuk evaluasi dan siap dipindahkan ke Raspberry Pi!")

## Cell 20 — Validasi Model NCNN

Cell ini mengukur performa **model NCNN** pada data `valid`. Data ini tidak dipakai untuk memperbarui bobot, tetapi dipakai selama training untuk memantau perkembangan dan memilih bobot terbaik, jadi hasilnya menggambarkan performa model pada data yang belum dilatihkan padanya.

Ada lima metrik yang dihitung:
- **Precision**: dari semua objek yang diprediksi model, berapa bagian yang benar. Nilai tinggi berarti sedikit salah deteksi.
- **Recall**: dari semua objek yang sebenarnya ada, berapa bagian yang berhasil ditemukan. Nilai tinggi berarti sedikit objek yang terlewat.
- **F1-Score**: gabungan Precision dan Recall dalam satu angka, dihitung dengan rumus `2 × Precision × Recall / (Precision + Recall)`. Nilainya tinggi hanya kalau Precision dan Recall sama-sama tinggi.
- **mAP50**: rata-rata *average precision* semua kelas dengan syarat kotak prediksi tumpang tindih minimal 50% (IoU 0,5) dengan kotak sebenarnya.
- **mAP50-95**: rata-rata mAP pada beberapa batas IoU dari 0,5 sampai 0,95. Syaratnya lebih ketat, jadi nilainya lebih rendah dari mAP50 dan lebih menggambarkan ketepatan posisi kotak.

Nilai Precision, Recall, F1-Score, dan mAP dicetak sebagai rata-rata dari semua kelas, sementara log Ultralytics menampilkan tabel per kelas. Setelah pembagian ulang di Cell 10, seluruh kelas punya contoh di data `valid`, jadi rata-rata ini mencakup semua kelas. Setelah itu grafik evaluasi (confusion matrix, kurva PR, dan lainnya) ditampilkan langsung di notebook lewat fungsi dari Cell 2. Hasilnya disimpan di folder `validation` di dalam `PROJECT_DIR`.

Ukuran gambar (`imgsz`) mengikuti ukuran export di `EXPORT_CONFIG`, karena model NCNN hanya menerima ukuran itu. Evaluasi berjalan satu gambar per langkah di CPU, jadi bisa memakan beberapa menit.

In [ ]:
val_results = eval_model.val(
    data    = str(clean_yaml_path),
    split   = "val",
    imgsz   = EXPORT_CONFIG["imgsz"],
    device  = "cpu",
    project = str(PROJECT_DIR),
    name    = "validation",
    plots   = True
)

val_p = float(val_results.box.mp)
val_r = float(val_results.box.mr)

val_metrics = {
    "Precision": val_p,
    "Recall"   : val_r,
    "F1-Score" : 2 * val_p * val_r / (val_p + val_r + 1e-9),
    "mAP50"    : float(val_results.box.map50),
    "mAP50-95" : float(val_results.box.map)
}

print("=== VALIDATION (MODEL NCNN) ===")
for name, value in val_metrics.items():
    print(f"  {name:10s}: {value:.4f}")

tampilkan_gambar_evaluasi(
    Path(val_results.save_dir),
    "Gambar Evaluasi Validation"
)

## Cell 21 — Evaluasi Final pada Test Set

Cell ini adalah pengujian akhir. Model NCNN dievaluasi pada data `test`, yaitu data yang tidak pernah dipakai untuk melatih model maupun untuk memilih bobot terbaik. Karena itu hasilnya adalah gambaran paling jujur tentang kemampuan model pada gambar baru, dan inilah angka yang sebaiknya dilaporkan sebagai performa akhir.

Metrik yang dihitung sama dengan validasi: Precision, Recall, F1-Score, mAP50, dan mAP50-95. Nilainya dicetak sebagai rata-rata semua kelas, dan Ultralytics juga menampilkan hasil per kelas di log sehingga terlihat kelas mana yang sudah baik dan kelas mana yang masih lemah. Setelah pembagian ulang di Cell 10, data `test` memuat contoh dari semua kelas, jadi tidak ada kelas yang nilainya kosong. Gambar yang diberi bounding box otomatis di Cell 11 tidak ada di data `test`, sehingga seluruh label di sini adalah label asli dataset.

Grafik evaluasi test, seperti confusion matrix dan kurva PR, ditampilkan di notebook dan disimpan di folder `final_test` di dalam `PROJECT_DIR`. Membandingkan hasil test dengan hasil validasi di Cell 20 berguna untuk melihat apakah performa model stabil. Kalau selisihnya jauh, model mungkin kurang bisa menyesuaikan diri dengan data baru.

In [ ]:
test_results = eval_model.val(
    data    = str(clean_yaml_path),
    split   = "test",
    imgsz   = EXPORT_CONFIG["imgsz"],
    device  = "cpu",
    project = str(PROJECT_DIR),
    name    = "final_test",
    plots   = True
)

test_p = float(test_results.box.mp)
test_r = float(test_results.box.mr)

test_metrics = {
    "Precision": test_p,
    "Recall"   : test_r,
    "F1-Score" : 2 * test_p * test_r / (test_p + test_r + 1e-9),
    "mAP50"    : float(test_results.box.map50),
    "mAP50-95" : float(test_results.box.map)
}

print("=== FINAL TEST SET (MODEL NCNN) ===")
for name, value in test_metrics.items():
    print(f"  {name:10s}: {value:.4f}")

tampilkan_gambar_evaluasi(
    Path(test_results.save_dir),
    "Gambar Evaluasi Final - Test Set"
)

## Cell 22 — Metrik per Kelas & Akurasi Deteksi (Test Set)

Rata-rata semua kelas bisa menyembunyikan kelas yang lemah. Cell ini menampilkan **tabel metrik untuk setiap kelas** pada data `test`: Precision, Recall, F1-Score, mAP50, dan mAP50-95. Dari tabel ini terlihat jelas kelas mana yang sudah dikenali dengan baik dan kelas mana yang masih sering salah, yang berguna untuk bahasan di laporan skripsi. Kelas yang tidak punya objek di data test (kalau ada) ditandai "tidak ada data" dan tidak dihitung.

Setelah itu cell ini menghitung **akurasi deteksi** dari confusion matrix. Akurasi biasa seperti pada klasifikasi tidak bisa dipakai begitu saja pada deteksi objek, karena model harus menebak kelas sekaligus posisi kotaknya. Di sini akurasi dihitung dengan cara berikut:

`Akurasi = TP / (TP + salah kelas + FP + FN)`

- **TP**: objek yang kotaknya cocok dan kelasnya benar.
- **Salah kelas**: kotaknya cocok dengan objek yang ada, tetapi kelasnya tertukar.
- **FP**: model mendeteksi sesuatu di tempat yang tidak ada objeknya.
- **FN**: objek yang ada tetapi tidak terdeteksi.

Angka ini bersifat pelengkap, bukan pengganti mAP. Confusion matrix dihitung Ultralytics dengan ambang confidence tertentu, sehingga nilainya bergantung pada ambang itu dan pada versi Ultralytics yang dipakai. Metrik utama untuk laporan tetap Precision, Recall, F1-Score, mAP50, dan mAP50-95.

In [ ]:
if "test_results" not in globals():
    print("Hasil test belum tersedia. Jalankan Cell 21 terlebih dahulu.")
else:
    box = test_results.box
    indeks_kelas = [int(i) for i in box.ap_class_index]
    posisi = {cid: pos for pos, cid in enumerate(indeks_kelas)}

    print(f"{'ID':<4}{'Kelas':<32}{'Precision':>11}{'Recall':>10}{'F1':>10}{'mAP50':>10}{'mAP50-95':>11}")
    print("-" * 88)

    for cid in range(NUM_CLASSES):
        if cid not in posisi:
            print(f"{cid:<4}{CLASS_NAMES[cid]:<32}{'tidak ada data':>52}")
            continue
        pos = posisi[cid]
        p, r = float(box.p[pos]), float(box.r[pos])
        f1 = 2 * p * r / (p + r + 1e-9)
        print(f"{cid:<4}{CLASS_NAMES[cid]:<32}{p:>11.4f}{r:>10.4f}{f1:>10.4f}"
              f"{float(box.ap50[pos]):>10.4f}{float(box.ap[pos]):>11.4f}")

    print("-" * 88)
    print(f"{'RATA-RATA':<36}{test_metrics['Precision']:>11.4f}{test_metrics['Recall']:>10.4f}"
          f"{test_metrics['F1-Score']:>10.4f}{test_metrics['mAP50']:>10.4f}{test_metrics['mAP50-95']:>11.4f}")

    # ---------- Akurasi deteksi dari confusion matrix ----------
    cm = getattr(test_results, "confusion_matrix", None)
    if cm is None:
        print("\nConfusion matrix tidak tersedia pada hasil evaluasi ini.")
    else:
        m  = np.array(cm.matrix, dtype=float)    # baris = prediksi, kolom = sebenarnya, indeks terakhir = latar belakang
        nc = NUM_CLASSES
        tp          = float(np.trace(m[:nc, :nc]))
        salah_kelas = float(m[:nc, :nc].sum() - tp)
        fp          = float(m[:nc, nc].sum())
        fn          = float(m[nc, :nc].sum())
        akurasi     = tp / (tp + salah_kelas + fp + fn + 1e-9)

        test_metrics["Akurasi"] = akurasi

        print("\n=== AKURASI DETEKSI (dari confusion matrix, test set) ===")
        print(f"  TP          : {int(tp)}")
        print(f"  Salah kelas : {int(salah_kelas)}")
        print(f"  FP          : {int(fp)}")
        print(f"  FN          : {int(fn)}")
        print(f"  Akurasi     : {akurasi:.4f}")

## Cell 23 — Prediksi pada Beberapa Gambar Test (Model NCNN)

Metrik memberi angka, sedangkan cell ini memperlihatkan langsung bagaimana model NCNN mendeteksi komponen pada gambar. Notebook mengambil 12 gambar pertama dari folder `test` pada dataset bersih (diurutkan berdasarkan nama file, bukan dipilih acak). Kalau gambar test kurang dari 12, semuanya dipakai.

Model NCNN punya ukuran input dan batch yang tetap (1 gambar per langkah), jadi gambar diprediksi **satu per satu** di dalam perulangan, di CPU, dengan `imgsz` yang sama dengan ukuran export. Cara ini sama dengan yang akan terjadi di Raspberry Pi, yaitu satu frame kamera per prediksi.

Prediksi dijalankan dengan `conf=0.25`, artinya hanya deteksi dengan confidence minimal 25% yang ditampilkan, sedangkan deteksi yang lebih ragu dibuang. Hasil prediksi juga disimpan sebagai gambar di folder `prediction_samples` di dalam `PROJECT_DIR`.

Untuk setiap gambar, notebook menampilkan:
- gambar dengan bounding box, nama kelas, dan nilai confidence di setiap objek yang terdeteksi,
- jumlah objek yang terdeteksi,
- daftar tiap objek beserta nama kelas dan confidence-nya, atau keterangan bahwa tidak ada objek yang terdeteksi.

Dari tampilan ini kita bisa menilai apakah kotaknya pas, apakah ada objek yang terlewat, dan apakah ada kelas yang sering tertukar.

In [ ]:
test_images_path = CLEAN_DATASET_DIR / "test" / "images"

test_images = sorted([
    p for p in test_images_path.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
])

print("Jumlah gambar test yang ditemukan:", len(test_images))

jumlah_gambar_prediksi = min(12, len(test_images))
sample_images          = test_images[:jumlah_gambar_prediksi]

print(f"Jumlah gambar yang digunakan untuk visualisasi: {jumlah_gambar_prediksi}")

# Model NCNN menerima satu gambar per langkah, jadi diprediksi satu per satu
for i, image_path in enumerate(sample_images, start=1):
    result = eval_model.predict(
        source   = str(image_path),
        imgsz    = EXPORT_CONFIG["imgsz"],
        conf     = 0.25,
        device   = "cpu",
        save     = True,
        project  = str(PROJECT_DIR),
        name     = "prediction_samples",
        exist_ok = True,
        verbose  = False
    )[0]

    hasil = result.plot()
    hasil = hasil[:, :, ::-1]

    plt.figure(figsize=(8, 8))
    plt.imshow(hasil)
    plt.axis("off")
    plt.title(f"Prediksi {i}: {image_path.name}")
    plt.show()

    print(f"Gambar {i}: {image_path.name}")
    print("Jumlah objek terdeteksi:", len(result.boxes))

    if len(result.boxes) > 0:
        for j, box in enumerate(result.boxes, start=1):
            class_id   = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = CLASS_NAMES[class_id]
            print(f"  Objek {j}: {class_name} (confidence={confidence:.3f})")
    else:
        print("  Tidak ada objek yang terdeteksi.")

    print("-" * 60)

## Cell 24 — Ringkasan Hasil

Cell terakhir merangkum seluruh hasil dalam satu tampilan supaya mudah disalin ke laporan atau dokumentasi. Isinya:
- model dasar yang dipakai (YOLOv8n),
- ringkasan dataset bersih: jumlah gambar dan objek di tiap bagian, serta jumlah gambar yang diberi bounding box otomatis atau dikeluarkan,
- konfigurasi training dan export: jumlah epoch maksimum, patience (early stopping), ukuran gambar, dan batch size,
- hasil validasi dan hasil test set: Precision, Recall, F1-Score, mAP50, dan mAP50-95 (plus akurasi deteksi untuk test set),
- lokasi folder model NCNN yang siap dibawa ke Raspberry Pi 4, beserta langkah singkat memakainya di sana.

Setiap bagian punya pengecekan sendiri. Kalau sebuah cell sebelumnya belum dijalankan, bagian yang bersangkutan menampilkan pesan yang meminta cell itu dijalankan, tidak error.

Kecepatan model (FPS) di Raspberry Pi 4 tidak bisa diukur dari laptop, jadi ukur langsung di Pi setelah folder model dipindahkan. Kalau kecepatannya kurang, perkecil `imgsz` di `EXPORT_CONFIG` (Cell 16) lalu jalankan ulang dari Cell 19. Kalau akurasi di kamera Pi lebih rendah dari angka di notebook ini, penyebab yang paling umum adalah perbedaan kualitas gambar antara foto dataset dan gambar dari Pi Camera Rev 1.3.

In [ ]:
print("=" * 70)
print("RINGKASAN YOLOv8n - ELECTROCOM61")
print("(Model NCNN untuk Raspberry Pi 4 + Pi Camera Rev 1.3)")
print("=" * 70)

print("\nModel Dasar   : YOLOv8n (Nano)")

print("\nDataset Bersih:")
if "ringkasan_data" in globals() and ringkasan_data["gambar"]:
    for split in splits:
        print(f"  {split:<6}: {ringkasan_data['gambar'][split]} gambar, {ringkasan_data['objek'][split]} objek")
    print("  Bbox otomatis (masuk ke train)  :", len(pseudo_items))
    print("  Gambar tanpa bbox yang dikeluarkan:", len(pool_kosong) - len(pseudo_items))
else:
    print("  Ringkasan dataset belum tersedia.")
    print("  Jalankan kembali Cell 13.")

print("\nKonfigurasi Training:")
print("  Epoch      :", TRAIN_CONFIG["epochs"])
print("  Image Size :", TRAIN_CONFIG["imgsz"])
print("  Batch Size :", TRAIN_CONFIG["batch"])
print("  Patience   :", TRAIN_CONFIG["patience"], "(early stopping)")

print("\nKonfigurasi Export:")
print("  Format     :", EXPORT_CONFIG["format"])
print("  Image Size :", EXPORT_CONFIG["imgsz"])
print("  Half (FP16):", EXPORT_CONFIG["half"])

print("\nHasil Validation (model NCNN):")
if "val_metrics" in globals():
    for name, value in val_metrics.items():
        print(f"  {name:10s}: {value:.4f}")
else:
    print("  Metrik Validation belum tersedia.")
    print("  Jalankan kembali Cell 20.")

print("\nHasil Test Set (model NCNN):")
if "test_metrics" in globals():
    for name, value in test_metrics.items():
        print(f"  {name:10s}: {value:.4f}")
else:
    print("  Metrik Test Set belum tersedia.")
    print("  Jalankan kembali Cell 21.")

print("\nFile Model Siap:")
if "dest_ncnn" in globals() and dest_ncnn.exists():
    print(f"  ✅ {dest_ncnn.resolve()}")
    print(f"  ✅ {classes_path.resolve()}")
    print("\nCara memakai di Raspberry Pi 4:")
    print("  1. Salin folder models/best_ncnn_model ke Raspberry Pi.")
    print("  2. Pasang paket di Pi:  pip install ultralytics ncnn")
    print("  3. Muat model        :  YOLO('models/best_ncnn_model', task='detect')")
    print(f"  4. Prediksi          :  model.predict(frame, imgsz={EXPORT_CONFIG['imgsz']}, conf=0.25)")
else:
    print("  ⚠️  Jalankan Cell 19 untuk mengekspor model ke NCNN.")

print("\n" + "=" * 70)
print("RINGKASAN SELESAI")
print("=" * 70)